# A flight plan over a valley near Hosín: GSD, overlaps, terrain and wind

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/flight_planner_hluboka_en.ipynb)

**OpenSkyLab · notebook for the lab** [Drone flight planner](https://openskylab.space/edu_viz_flight_planner.html) · [česká verze](flight_planner_hluboka_cs.ipynb)

In the lab you planned a flight over flat ground and over a model hill. Here you apply the same formulas to real terrain: a wooded valley with a railway line near Hosín, about 3 km east of Hluboká nad Vltavou from the ČÚZK DMR 5G digital terrain model (lidar, 2 m cells), and to the current wind forecast at flight height.

**Goal:** compute flight height for a target GSD, line spacing, trigger interval and speed, motion blur and skew, show how terrain changes height above ground and overlap, and plan the return against the wind.

**Data:** DMR 5G (ČÚZK, ImageServer, heights in Bpv, 2 m cells) for a 1.2 × 0.8 km area in a valley near Hosín east of Hluboká nad Vltavou; hourly wind forecast at 10, 80 and 120 m from Open-Meteo; camera parameters from manufacturers' data sheets (same as in the lab).

**Tasks:** five. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 50 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

> **💻 What the code does:** Optional cell. If you paste the settings copied from the lab and run it, the cells below take the same parameter values and say so for each one. If you leave it empty, the default values are used. It downloads nothing.

In [ ]:
# @title Settings from the lab (optional)
# @markdown In the lab, click the Notebook link. The settings are copied; paste them here in place of `{}` (Ctrl+V) and run the cell.
Z_VIZUALU = {}  # @param {type:"raw"}
STEM = "flight_planner"


def z_viz(name, default, mapping=None):
    """Value of a parameter from the lab if the pasted code contains it; otherwise the default value."""
    z = (
        Z_VIZUALU
        if isinstance(Z_VIZUALU, dict) and Z_VIZUALU.get("osl") == STEM
        else {}
    )
    if name not in z:
        return default
    v = z[name]
    if mapping is not None:
        if v not in mapping:
            print(
                f"{name}: value {v!r} from the lab is not in this notebook, keeping {default!r}"
            )
            return default
        v = mapping[v]
    elif isinstance(default, bool):
        v = bool(v)
    elif isinstance(default, int):
        v = int(round(v))
    elif isinstance(default, float):
        v = float(v)
    print(f"{name} = {v!r}  (from the lab)")
    return v


if isinstance(Z_VIZUALU, dict) and Z_VIZUALU and Z_VIZUALU.get("osl") != STEM:
    print(
        f"The pasted code is from another lab ({Z_VIZUALU.get('osl')}), the notebook does not use it."
    )

## 1. Setup

> **💻 What the code does:** Installs the rasterio library, which reads raster data (GeoTIFF) directly from the internet. The other libraries are already in Colab.

In [ ]:
%pip install -q rasterio

> **💻 What the code does:** Loads the libraries, defines `cuzk_raster`, which downloads a window of a ČÚZK raster (DMR 5G terrain or DMP 1G surface model) as an array of elevations, and `open_meteo`, which queries the weather and retries after an error. It stores the parameters of four cameras: sensor size, pixel count, focal length and the shortest interval between photos.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from pyproj import Transformer


def cuzk_raster(service, x0, y0, x1, y1, res=2.0):
    """Window of a ČÚZK raster (dmr5g, dmp1g, dmp_obrazova_korelace) in S-JTSK (EPSG:5514), heights in Bpv.
    Returns the height array (row 0 = north) and its transform."""
    nx, ny = int(round((x1 - x0) / res)), int(round((y1 - y0) / res))
    url = f"https://ags.cuzk.cz/arcgis2/rest/services/{service}/ImageServer/exportImage"
    params = {
        "bbox": f"{x0},{y0},{x1},{y1}",
        "bboxSR": 5514,
        "imageSR": 5514,
        "size": f"{nx},{ny}",
        "format": "tiff",
        "pixelType": "F32",
        "interpolation": "RSP_BilinearInterpolation",
        "f": "image",
    }
    for attempt in range(4):
        r = requests.get(url, params=params, timeout=120)
        if r.ok and r.headers.get("content-type", "").startswith("image"):
            break
        time.sleep(3 * (attempt + 1))
    r.raise_for_status()
    with rasterio.MemoryFile(r.content) as mem, mem.open() as ds:
        return ds.read(1).astype(float), ds.transform


def open_meteo(params, tries=6):
    """Query Open-Meteo; retry on errors or when the daily limit is exceeded."""
    for i in range(tries):
        try:
            r = requests.get(
                "https://api.open-meteo.com/v1/forecast", params=params, timeout=60
            )
            if r.ok:
                return r.json()
            reason = r.json().get("reason", r.status_code)
        except requests.RequestException as e:
            reason = type(e).__name__
        print("Open-Meteo:", reason, "– retrying")
        time.sleep(5 * (i + 1))
    raise RuntimeError("Open-Meteo is not responding, run the cell again in a while.")


# cameras: sensor width and height (mm), pixel count, focal length (mm), shortest interval between shots (s)
CAMERAS = {
    "P1 35 mm": dict(sw=35.9, sh=24.0, pw=8192, ph=5460, f=35.0, tmin=0.7),
    "Mavic 3E": dict(sw=17.3, sh=13.0, pw=5280, ph=3956, f=12.3, tmin=0.7),
    "Mavic 3M (MS)": dict(sw=5.2, sh=3.9, pw=2592, ph=1944, f=4.34, tmin=2.0),
    "RGB 1″ 20 Mpx": dict(sw=13.2, sh=8.8, pw=5472, ph=3648, f=8.8, tmin=2.0),
}

## 2. Camera, height and GSD

GSD = H · p / f, where p is the pixel pitch on the sensor (sensor width / pixel count). From a target GSD, H = GSD · f / p.

> **💻 What the code does:** For the target GSD `gsd_target` it computes, for each camera, the pixel size on the sensor, the required flight height and the image footprint on the ground. It shows a table.

In [ ]:
gsd_target = 1.5  # ◀ parameter: target GSD in cm/px
rows = []
for name, c in CAMERAS.items():
    p = c["sw"] / c["pw"]  # mm
    H = gsd_target / 100 * c["f"] / p  # m
    rows.append(
        {
            "camera": name,
            "pixel pitch (µm)": p * 1000,
            "flight height (m)": H,
            "footprint across (m)": c["pw"] * gsd_target / 100,
            "footprint along (m)": c["ph"] * gsd_target / 100,
        }
    )
cam_tab = pd.DataFrame(rows).set_index("camera")
cam_tab.round(1)

> **✏️ Task 1.** Why does the Mavic 3M have to fly much lower than the P1 for the same GSD? Set `gsd_target` to 2 and then 4 cm/px. For which cameras and GSDs would the height exceed 120 m, the open-category limit? What are the consequences of flying low (number of images, time, safety)?

*Your answer:* 

## 3. Real terrain and the line plan

We load DMR 5G for the valley near Hosín. Lines run east–west, i.e. down the fall line and back up. The take-off point is in the field at the upper, western edge.

> **💻 What the code does:** Downloads the ČÚZK DMR 5G terrain model with a 2 m cell for a 1200 × 800 m area, determines the take-off elevation on the western edge and plots the terrain with 10 m contours.

In [ ]:
cx, cy = Transformer.from_crs(4326, 5514, always_xy=True).transform(14.4769, 49.0452)
W, Hh = 1200, 800  # area: width (E–W) and height (N–S) in m
dem, tr = cuzk_raster("dmr5g", cx - W / 2, cy - Hh / 2, cx + W / 2, cy + Hh / 2, res=2)
z_takeoff = float(np.median(dem[:, :10]))  # upper, western edge of the area
print(
    f"terrain {dem.min():.1f}–{dem.max():.1f} m a.s.l. (Bpv), take-off point {z_takeoff:.1f} m a.s.l."
)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(dem, cmap="terrain", extent=(0, W, 0, Hh))
fig.colorbar(im, label="elevation (m, Bpv)")
ax.contour(
    np.flipud(dem),
    levels=np.arange(np.floor(dem.min() / 10) * 10, dem.max() + 10, 10),
    colors="k",
    linewidths=0.4,
    extent=(0, W, 0, Hh),
)
ax.set(xlabel="m (west → east)", ylabel="m", title="DMR 5G, valley near Hosín")
plt.show()

> **💻 What the code does:** For the chosen camera, height, overlaps and speed (they can come from the lab) it computes the GSD, the distance between photos (base), the line spacing, the number of lines and photos, the photo interval and the flight time. It warns if the camera cannot keep up with the interval.

In [ ]:
# Camera.
cam = CAMERAS[
    z_viz(
        "cam",
        "Mavic 3E",
        {
            "rgb": "RGB 1″ 20 Mpx",
            "p1": "P1 35 mm",
            "m3e": "Mavic 3E",
            "m3m": "Mavic 3M (MS)",
        },
    )
]  # ◀ parameter
H0 = z_viz("H0", 80)  # ◀ parameter: flight height above the take-off point (m)
fo, so = z_viz("fo", 0.80), z_viz("so", 0.70)  # ◀ parameter: forward and side overlap
v_air = z_viz("v_air", 10.0)  # ◀ parameter: airspeed (m/s)
long_across = True  # ◀ parameter: long side of the image across the flight direction?

p = cam["sw"] / cam["pw"] / 1000  # m
n_across, n_along = (cam["pw"], cam["ph"]) if long_across else (cam["ph"], cam["pw"])
gsd0 = H0 * p / (cam["f"] / 1000)  # GSD above the take-off point
base = (1 - fo) * n_along * gsd0  # distance between shots
spacing = (1 - so) * n_across * gsd0  # line spacing
n_lines = int(np.ceil(Hh / spacing)) + 1
interval = base / v_air
photos = n_lines * (int(W / base) + 1)
t_flight = (n_lines * W + (n_lines - 1) * spacing) / v_air
print(
    f"GSD above take-off {gsd0 * 100:.2f} cm/px, base {base:.1f} m, line spacing {spacing:.1f} m"
)
print(
    f"lines {n_lines}, images {photos}, interval {interval:.2f} s (camera can do {cam['tmin']} s), flight time {t_flight / 60:.1f} min without turns"
)
if interval < cam["tmin"]:
    print(
        f"WARNING: the camera cannot keep up; the highest speed is {base / cam['tmin']:.1f} m/s"
    )

> **✏️ Task 2.** Increase `v_air` until the warning that the camera cannot keep up appears. What is the highest speed? Switch `long_across = False`: how do base, maximum speed, number of lines and flight time change? Why does a rotated image allow faster flight and yet the flight may take longer? Set the forward overlap to 0.85 as for vegetation.

*Your answer:* 

## 4. Constant height or terrain following?

At a constant height above the take-off point, the height above ground changes with terrain: H_AGL = H0 + z_takeoff − z. GSD and footprint change with it, while base and line spacing stay fixed, so the actual overlap changes.

> **💻 What the code does:** For each terrain cell it computes the height above ground, the GSD and the actual forward and side overlap, either at a constant height above the take-off point or when following the terrain (`follow`). It prints their ranges and the share of the area more than 120 m above ground and plots three maps.

In [ ]:
# Constant height H0 above ground.
follow = False  # ◀ parameter: True = terrain following
agl = np.full_like(dem, H0) if follow else H0 + z_takeoff - dem
gsd = agl * p / (cam["f"] / 1000)
fo_local = 1 - base / (n_along * gsd)
so_local = 1 - spacing / (n_across * gsd)
print(
    f"height above ground {agl.min():.0f}–{agl.max():.0f} m, GSD {gsd.min() * 100:.2f}–{gsd.max() * 100:.2f} cm/px"
)
print(
    f"forward overlap {fo_local.min():.0%}–{fo_local.max():.0%}, side overlap {so_local.min():.0%}–{so_local.max():.0%}"
)
print(f"area more than 120 m above terrain: {np.mean(agl > 120):.0%} of the area")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, (arr, lbl, cm) in zip(
    ax,
    [
        (agl, "height above ground (m)", "viridis"),
        (gsd * 100, "GSD (cm/px)", "magma"),
        (so_local * 100, "side overlap (%)", "RdYlGn"),
    ],
):
    im = a.imshow(arr, cmap=cm, extent=(0, W, 0, Hh))
    fig.colorbar(im, ax=a, label=lbl)
    a.set_xticks([])
    a.set_yticks([])
ax[0].set_title("terrain following" if follow else f"constant {H0} m above take-off")
plt.show()

> **✏️ Task 3.** At a constant 80 m above the take-off point: what is the range of GSD and overlaps? Where is the overlap smallest, at the top near the take-off point or in the lowest places, and why? How much of the area is more than 120 m above the terrain? Switch to `follow = True`: what improves and what do you need for terrain following (terrain model, its accuracy, obstacles)?

*Your answer:* 

## 5. Real wind, motion blur and rolling shutter

We take today's wind forecast for this area from Open-Meteo. Blur = ground speed · exposure / GSD; rolling-shutter skew = ground speed · sensor readout time / GSD. The Zenmuse P1 and Mavic 3E have a mechanical shutter, so skew practically disappears for them (t_read = 0); the default 30 ms is an order-of-magnitude value for an electronic rolling shutter.

> **💻 What the code does:** Downloads today's hourly forecast from Open-Meteo for the area: wind at 10, 80 and 120 m, wind direction at 80 m and gusts, and prints the hours from 6 to 19.

In [ ]:
w = open_meteo(
    {
        "latitude": 49.0452,
        "longitude": 14.4769,
        "forecast_days": 1,
        "wind_speed_unit": "ms",
        "timezone": "Europe/Prague",
        "hourly": "wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_direction_80m,wind_gusts_10m",
    }
)
met = pd.DataFrame(w["hourly"]).set_index("time")
print(met.iloc[6:20].to_string())

> **💻 What the code does:** For the chosen hour it splits the wind at 80 m into components along and across the lines and computes the ground speed on the eastbound and westbound line. From it and the exposure `t_exp` and sensor readout time `t_read` (they can come from the lab) it prints the motion blur and the rolling-shutter skew in pixels.

In [ ]:
# Hour of the flight (local time); default is the windiest hour between 6 and 20 h.
hour = 6 + int(met.wind_speed_80m.values[6:21].argmax())  # ◀ parameter
t_exp = z_viz("t_exp", 1 / 1000)  # ◀ parameter: exposure time (s)
# Sensor readout time of a rolling shutter (s); 0 = global shutter.
t_read = z_viz("t_read", 0.030)  # ◀ parameter
wind = met.wind_speed_80m.iloc[hour]
wdir = met.wind_direction_80m.iloc[hour]
# lines run E–W; wind component along and across the lines
along = wind * np.cos(np.radians(wdir - 90))
cross = wind * np.sin(np.radians(wdir - 90))
for leg, sign in [("eastbound line", -1), ("westbound line", +1)]:
    # Ground speed (wind is "from" → sign).
    gs = np.sqrt(max(v_air**2 - cross**2, 0)) + sign * along
    print(
        f"{leg}: ground speed {gs:4.1f} m/s, blur {gs * t_exp / gsd0:.2f} px, skew {gs * t_read / gsd0:.1f} px"
    )
print(
    f"{met.index[hour]}: wind at 80 m: {wind:.1f} m/s from {wdir:.0f}°; eastward component {-along:+.1f} m/s, northward {cross:+.1f} m/s"
)

> **✏️ Task 4.** What wind does the forecast give at 10, 80 and 120 m and why do they differ? How do blur and skew differ on the downwind and upwind legs? Why does a shorter exposure not reduce the skew? When would you rather orient the lines along or across the wind?

*Your answer:* 

## 6. Battery and return to home

The Mavic 3E has a 77 Wh battery and the maker states up to 45 min of flight without wind, i.e. about 100 W on average. We compute how much energy a return from the farthest point needs when the wind blows away from the take-off point.

> **💻 What the code does:** Estimates the energy for the mission from the drone's average power and for the return from the farthest corner of the area against the wind. It compares their sum with the usable battery energy after temperature and reserve and prints whether the battery is enough.

In [ ]:
E_bat = 77.0  # ◀ parameter: battery energy (Wh)
P = 77.0 / (45 / 60)  # average power (W) from the stated endurance
# 1.0 at 20 °C; roughly 0.7–0.8 in frost.
temp_factor = 1.0  # ◀ parameter: usable capacity
reserve = 0.20  # ◀ parameter: landing reserve
headwind = wind  # worst case: all of the wind against the return
d_home = np.hypot(W, Hh)  # farthest corner of the area (m)
gs_home = v_air - headwind
E_mission = P * t_flight / 3600
E_return = P * d_home / max(gs_home, 0.1) / 3600
E_use = E_bat * temp_factor * (1 - reserve)
print(
    f"mission {E_mission:.1f} Wh + return {E_return:.1f} Wh (headwind {headwind:.1f} m/s, ground speed {gs_home:.1f} m/s) = {E_mission + E_return:.1f} Wh"
)
print(
    f"usable energy {E_use:.1f} Wh → {'enough' if E_mission + E_return <= E_use else 'NOT ENOUGH, split the mission'}"
)

> **✏️ Task 5.** Is one battery enough? Set `temp_factor = 0.75` and then increase the wind (`headwind = 8`). Why is "downwind out, upwind back" a trap? Also try a higher `v_air`: why does flying faster help in strong wind even though power increases?

*Your answer:* 

> **✏️ Extension (optional).** Classmates may have the notebook for another area (Roztoky, Brno-Bystrc, Hosín near Hluboká). Compare the relief, the range of height above ground at a constant 80 m above the take-off point and the share of the area above 120 m. Where would a constant height be least suitable, and how many batteries would each of you need for the area?

*Your answer:* 

---
### Data sources and references

- Digital terrain model of the Czech Republic, 5th generation (DMR 5G), © ČÚZK, licence CC BY 4.0, service https://ags.cuzk.cz/arcgis2/rest/services/dmr5g/ImageServer
- Weather: Open-Meteo.com (CC BY 4.0), https://open-meteo.com
- Camera parameters: data sheets of DJI Zenmuse P1, Mavic 3 Enterprise (3E, 3M) and a generic 1″ 20 MP camera (the shortest interval of 2 s for the Mavic 3M and the generic camera is an indicative value); Mavic 3 Enterprise battery 5000 mAh, 15.4 V (77 Wh).
- Flight height in the open category max. 120 m from the closest point of the surface of the earth: Commission Implementing Regulation (EU) 2019/947, Annex, UAS.OPEN.010.

### How to cite

OpenSkyLab (2026). A flight plan over a valley near Hosín: GSD, overlaps, terrain and wind [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/flight_planner_hluboka_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*